# Tutorial — Modelos econométricos (satélite) de PD, LGD e CCF (`yggdrasil.credit_risk.econometric`)

Os **modelos econométricos** (ou **satélite**, ou **macro**) ligam as **séries temporais agregadas** dos
parâmetros de risco — taxa de *default* (PD), severidade (LGD) e fator de conversão (CCF) por segmento — às
**variáveis macroeconômicas** (desemprego, renda, juros, câmbio, inadimplência) e respondem à pergunta central:

> *Dado um cenário macroeconômico, quanto valem a PD, a LGD e o CCF deste segmento nos próximos trimestres?*

Essa capacidade de **projeção condicional** alimenta quatro usos:

1. o **forward-looking do ECL** sob IFRS 9 / Resolução CMN 4.966 (o parâmetro PIT incorpora cenários futuros ponderados);
2. os **testes de estresse** (traduzir um cenário adverso em perdas);
3. o **capital econômico** — a ligação entre fator sistêmico e macro é, ela mesma, um modelo satélite (ver `credit_risk.capital`);
4. o **planejamento** de provisão e apetite de risco.

É o **eixo temporal**, complementar ao **eixo transversal** da escoragem (`credit_risk.model`/`tree`): o modelo
transversal **ordena** o risco entre clientes; o satélite **desloca o nível** da curva conforme o ciclo.

Este tutorial percorre o método na ordem em que o guia o constrói: **séries → transformações → diagnóstico →
modelo principal (ARDL) → seleção → projeção por cenários → LGD/CCF → fator Z de Vasicek → extensões →
pipeline declarativo e governança**.

> **Dependências:** o subpacote usa `statsmodels` + `arch` (extra `econometric`: `pip install -e ".[econometric]"`).
> O restante de `credit_risk` (capital, segmentadores) **não** os exige — o subpacote é carregado sob demanda.

## 0. Setup

Núcleo em `statsmodels`/`arch`; figuras em `matplotlib`; o registro no MLflow (seção 10) usa o *file store* local.

In [ ]:
%matplotlib inline
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")   # MLflow local (seção 10)

import warnings; warnings.filterwarnings("ignore")          # silencia avisos de convergência do statsmodels
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 40)

import yggdrasil.credit_risk.econometric as E
print("econometric — API com", len(E.__all__), "símbolos públicos")

## 1. As séries: contrato de dados e gerador sintético

O objeto que percorre a biblioteca é a **`RiskSeries`**: a série temporal agregada de um parâmetro de risco de
um **segmento homogêneo**, com `kind` (`'pd'`, `'lgd'` ou `'ccf'`), frequência e índice temporal.

Como não temos aqui as bases contratuais internas, usamos o **gerador sintético** — que o guia chama de *"o
investimento de melhor retorno do plano"*: séries de **processo gerador conhecido** (ARDL em logit / fator Z de
Vasicek, dirigidas por macro e um choque sistêmico) permitem a única validação forte possível para código
econométrico: **testar se cada método recupera a verdade**.

`make_reference_study()` monta o **estudo de referência**: uma macro única (com recessão e pandemia) dirigindo
as três séries — PD, LGD e CCF — do mesmo segmento e horizonte.

In [ ]:
est = E.make_reference_study(n_periods=120, seed=7)
macro = est.macro
print("Macro:", list(macro.columns), "| períodos:", len(macro))
print("Séries:", est.pd.series.kind, est.lgd.series.kind, est.ccf.series.kind,
      "| médias:", round(est.pd.series.values.mean(), 4),
      round(est.lgd.series.values.mean(), 3), round(est.ccf.series.values.mean(), 3))
macro.tail(3)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), height_ratios=[1, 1], sharex=True)
macro[["desemprego", "juros"]].plot(ax=ax[0], title="Variáveis macro (com recessão 2016-17 e pandemia 2020)")
ax[0].grid(alpha=.25)
est.pd.series.values.plot(ax=ax[1], color="crimson", label="PD")
ax[1].set_title("Taxa de default (PD) do segmento"); ax[1].grid(alpha=.25); ax[1].legend()
fig.tight_layout(); fig

## 2. Transformações e o ritual de estacionariedade

Taxas vivem em $(0,1)$: modelá-las **em nível** viola as hipóteses lineares (previsões fora do intervalo,
variância não constante). As transformações padrão levam a taxa para a reta real:

$$\text{logit}(p)=\ln\frac{p}{1-p}, \qquad \text{probit}(p)=N^{-1}(p).$$

Uma terceira, específica de PD, é o **fator $Z$ de Vasicek** (seção 8): inverte a fórmula de Vasicek e extrai
o fator sistêmico latente.

Antes de qualquer regressão vem o **ritual obrigatório** (guia §2.3): estacionariedade por **ADF**, **KPSS** e
**Phillips-Perron** — usados **em conjunto** porque têm hipóteses nulas **opostas** (ADF: raiz unitária; KPSS:
estacionária).

In [ ]:
from yggdrasil.credit_risk.econometric import transforms as tf, diagnostics as diag

y_logit = tf.logit(est.pd.series.values)          # PD na escala de modelagem
rep = diag.stationarity_report(y_logit)
print("Ordem de integração estimada:", rep.attrs["ordem_integracao"])
rep[["teste", "estatistica", "p_valor", "ok", "conclusao"]]

## 3. O modelo principal: ARDL

O **cavalo de batalha** dos modelos satélite (guia §3.1). A PD transformada é regredida contra **defasagens dela
própria** (a inércia da inadimplência) e das **variáveis macro** defasadas:

$$y_t = c + \sum_i \varphi_i\, y_{t-i} + \sum_j \beta_j\, x_{j,\,t-\ell} + \varepsilon_t.$$

A `Specification` descreve o modelo de forma declarativa: quais macro/defasagens, ordem AR, *link*, dummies
sazonais/evento, e o **sinal econômico esperado** de cada variável. Ajustamos e rodamos a **bateria de
diagnóstico de resíduos** (Ljung-Box, Breusch-Godfrey/Pagan, White, Jarque-Bera, ARCH-LM, CUSUM).

In [ ]:
from yggdrasil.credit_risk.econometric import Specification, ARDL

spec = Specification(
    exog={"desemprego": [1], "renda": [0]},   # desemprego defasado 1m, renda contemporânea
    ar=1, link="logit",
    events={"covid": ("2020-03", "2020-08")}, # dummy de evento (a pandemia; guia §2.3)
    expected_signs={"desemprego": 1, "renda": -1},
)
m = ARDL(est.pd.series, macro, spec)
fit = m.fit()
print(f"AIC={fit.aic:.1f}  BIC={fit.bic:.1f}  R²={fit.rsquared:.3f}  n={fit.nobs}")
fit.coef_frame()

In [ ]:
# Bateria de diagnóstico de resíduos (guia §4.2)
fit.diagnostics()[["teste", "estatistica", "p_valor", "ok", "conclusao"]]

In [ ]:
# Ajuste observado x previsto + resíduos
fig = E.report.plot_fit(fit, est.pd.series); fig

## 4. Seleção champion-challenger

Em séries curtas, o ajuste in-sample engana: $R^2$ alto é barato. A seleção **pune complexidade** e **premia
desempenho fora da amostra e coerência econômica** (guia §4). `search` percorre uma grade parcimoniosa (2–4
variáveis, defasagens 0–6) e aplica **filtros duros**:

- **sinal econômico** coerente (coeficiente com sinal trocado é *desqualificador* — a projeção de estresse iria
  na direção errada);
- **VIF** controlado (colinearidade macro);
- **validação walk-forward** (RMSE fora da amostra) contra os **benchmarks ingênuos** e o **ARIMA** — um modelo
  macro que não supera o ARIMA fora da amostra não está pronto.

In [ ]:
res = E.search(
    est.pd.series, macro,
    candidates=["desemprego", "renda", "juros"],
    expected_signs={"desemprego": 1, "renda": -1, "juros": 1},
    grid_kwargs={"lag_set": (0, 1, 3), "max_vars": 2, "max_specs": 60},
    horizon=6, min_train=72, vif_max=8.0,
)
print("Melhor especificação:", res.best_spec.describe())
res.top(8)[["modelo", "status", "AIC", "max_vif", "oos_rmse", "vs_arima"]]

A coluna `vs_arima` é o RMSE fora da amostra **relativo ao ARIMA**: abaixo de 1 significa que o modelo macro
**supera** o benchmark univariado. O `Diebold-Mariano` testa se a diferença de acurácia é significativa.

In [ ]:
from yggdrasil.credit_risk.econometric.benchmarks import RandomWalk
wf_best = E.walk_forward(lambda s, mm: E.ARDL(s, mm, res.best_spec), est.pd.series, macro,
                         min_train=72, horizon=6)
wf_rw   = E.walk_forward(lambda s, mm: RandomWalk(s), est.pd.series, macro, min_train=72, horizon=6)
dm = E.diebold_mariano(wf_best["errors"], wf_rw["errors"], h=6)
print(f"RMSE  ARDL={wf_best['rmse']:.4f}  RandomWalk={wf_rw['rmse']:.4f}")
print(f"Diebold-Mariano: stat={dm['stat']:.2f}  p={dm['pvalue']:.4f}  (mais acurado: {dm['better']})")

## 5. Projeção por cenários — o produto final (Figura 1 do guia)

Com o modelo estimado, a projeção é mecânica: alimentam-se as **trajetórias macro** de cada cenário e o modelo
devolve a trajetória da PD, reconvertida à escala original, com **intervalos por simulação de resíduos** (os
erros se acumulam ao longo do horizonte — guia §5).

`standard_scenarios` monta base/adverso/otimista; `shock_scenarios` permite choques customizados. A **fonte
única** de cenários e projeção evita "números diferentes para a mesma pergunta em áreas diferentes".

In [ ]:
scen = E.standard_scenarios(macro, horizon=12, stress_var="desemprego",
                            probabilities=(0.5, 0.3, 0.2))   # base, otimista, adverso
proj = m.project(scen, n_sims=2000, seed=1, alpha=0.10)
proj.mean_frame().round(4)

In [ ]:
fig = E.report.plot_projection(proj, est.pd.series); fig

Para o **ECL forward-looking**, as projeções por cenário são **ponderadas pelas probabilidades** definidas na
governança:

$$\text{PD}^{ECL}_t = \sum_s w_s \cdot \text{PD}_{s,t}.$$

In [ ]:
ecl = E.ecl_projection(m, scen)
ecl.round(4).to_frame("PD_ECL").T

## 6. LGD e CCF: modelos de resposta limitada

Para **LGD** e **CCF** (frações em $[0,1]$, muitas vezes perto das bordas), o guia recomenda modelos que
respeitam o suporte **nativamente** (§3.5):

- **regressão beta** (`BetaRegression`) — modela média e dispersão da distribuição beta;
- **fractional logit** (`FractionalLogit`, Papke-Wooldridge) — GLM binomial com *link* logit e erros-padrão
  robustos; aceita observações **exatamente em 0 e 1** (curas totais / perdas totais).

Também é possível a **decomposição** LGD = (1 − prob. de cura) × severidade dos não curados (`decompose_lgd`).

In [ ]:
from yggdrasil.credit_risk.econometric import BetaRegression, FractionalLogit

spec_lgd = Specification(exog={"desemprego": [0], "cambio": [3]}, ar=1, link="logit")
mb = BetaRegression(est.lgd.series, macro, spec_lgd); fb = mb.fit()
mf = FractionalLogit(est.lgd.series, macro, spec_lgd); ff = mf.fit()
print("LGD — coeficientes (beta vs fractional logit):")
pd.concat({"beta": fb.params, "fractional": ff.params}, axis=1).round(4)

In [ ]:
# CCF — o fator de conversão do limite não sacado (utilização até o default)
spec_ccf = Specification(exog={"juros": [2], "desemprego": [1]}, ar=1, link="logit")
mc = FractionalLogit(est.ccf.series, macro, spec_ccf); fc = mc.fit()
print(f"CCF — modelo {fc.model_name}, AIC={fc.aic:.1f}")
proj_ccf = mc.project(E.standard_scenarios(macro, 12), n_sims=1000, seed=2)
proj_ccf.mean_frame().round(4).tail(3)

## 7. Fator $Z$ de Vasicek — a ponte com o capital econômico

A abordagem que **nasce do arcabouço de crédito** (§3.4): em vez de um *link* genérico, inverte-se a fórmula de
Vasicek para extrair o **fator sistêmico latente $Z$** (aproximadamente $N(0,1)$ e estacionário) e modela-se
$Z$ contra a macro. As projeções de $Z$ voltam a PD pela fórmula direta.

$$Z_t = \frac{N^{-1}(PD_{TTC}) - \sqrt{1-\rho}\, N^{-1}(DR_t)}{\sqrt{\rho}}.$$

Vantagem: **consistência total com o motor de capital** (`credit_risk.capital`) — o mesmo $Z$ alimenta a
simulação multifatorial. É a ponte formal entre este guia e o de capital.

> **Sinal:** como $Z$ alto = ciclo **benigno**, o desemprego entra com coeficiente **negativo** (mais desemprego
> ⇒ $Z$ menor ⇒ PD maior).

In [ ]:
from yggdrasil.credit_risk.econometric import VasicekZ

pd_ttc = est.pd.series.ttc()          # PD through-the-cycle (média de longo prazo)
mz = VasicekZ(est.pd.series, macro,
              Specification(exog={"desemprego": [1]}, ar=1),
              rho=0.12, pd_ttc=pd_ttc)
fz = mz.fit()
print(f"PD_TTC={pd_ttc:.4f}  rho=0.12  | coef desemprego_l1={fz.params['desemprego_l1']:.4f} (esperado < 0)")
fz.coef_frame()

## 8. Extensões: cointegração, VAR/VECM e painel

- **Cointegração** (Engle-Granger / Johansen) e **VECM**: quando a relação é de **equilíbrio de longo prazo**
  entre séries integradas (§3.3).
- **VAR**: trata tudo como endógeno — útil para **impulso-resposta** (quanto a PD responde a um choque de juros
  ao longo dos meses) e cenários internamente consistentes.
- **Painel de segmentos** (§3.7): com muitos segmentos de séries curtas, empilhar em painel com **efeitos fixos**
  e **inclinações macro comuns** aumenta o poder estatístico e **disciplina os sinais**.

In [ ]:
# Impulso-resposta da PD a choques macro (VAR)
vm = E.VARModel(est.pd.series, macro, variables=["desemprego", "juros"], maxlags=3).fit()
irf = vm.irf(periods=10)
fig, ax = plt.subplots(figsize=(9, 4))
irf[["choque_desemprego", "choque_juros"]].plot(ax=ax, marker="o", ms=3,
    title="Resposta da PD (escala logit) a um choque de 1 desvio")
ax.axhline(0, color="gray", lw=.8); ax.grid(alpha=.25); fig

In [ ]:
# Cointegração (Johansen) do sistema PD-macro
jt = E.johansen_test(vm.data)
print("Johansen — posto de cointegração estimado:", jt.rank)

# Painel: 6 segmentos sintéticos com a MESMA relação macro -> recupera coeficientes com mais poder
from yggdrasil.credit_risk.econometric import series as S, PanelSatellite
panels = {f"seg{i}": S.simulate_pd_series(macro=macro, seed=100 + i,
             betas=(("desemprego", 1, 0.18), ("renda", 0, -0.06))).series for i in range(6)}
pr = PanelSatellite(panels, macro, Specification(exog={"desemprego": [1], "renda": [0]}, ar=1)).fit()
print("\nPainel (6 segmentos) — inclinações comuns [verdade: desemprego_l1=0.18, renda=-0.06, AR=0.5]:")
pr.macro_frame()

## 9. Pipeline declarativo — as "cinco chamadas"

O guia fixa a régua de sucesso do desenho: **cinco chamadas** entre a base bruta e o relatório de governança
(§6.3). A `StudyConfig` descreve o estudo de forma **versionável**; `run_study` encadeia **seleção → ajuste →
diagnóstico → projeção por cenários → relatório**, opcionalmente registrando no MLflow.

In [ ]:
cfg = E.StudyConfig(
    kind="pd", name="pd_cartao_revolver",
    candidates=["desemprego", "renda", "juros"],
    expected_signs={"desemprego": 1, "renda": -1, "juros": 1},
    lag_set=(0, 1, 3), max_vars=2, max_specs=60,
    horizon=12, min_train=72, vif_max=8.0,
)
r = E.run_study(cfg, est.pd.series, macro, make_report=True)
r.summary()

In [ ]:
# O relatório de governança (HTML) sai pronto — com especificação, coeficientes, testes e o leque
print("Relatório HTML:", len(r.report_html), "bytes | seções:",
      [s for s in ("Coeficientes", "Métricas", "Diagnóstico", "Projeção") if s in r.report_html])
# E.report.save_report(r.report_html, "relatorio_pd_cartao.html")   # para gravar em disco

## 10. Registro no MLflow

Modelos satélite alimentam **provisão, estresse e capital** — a governança (§4.4) exige inventário,
versionamento e trilha de auditoria. `log_satellite_run` registra especificação, métricas, a bateria de
diagnóstico, a projeção e as figuras num *run* do MLflow.

In [ ]:
import mlflow, tempfile, pathlib
with tempfile.TemporaryDirectory() as d:
    mlflow.set_tracking_uri((pathlib.Path(d) / "mlruns").as_uri())
    run_id = E.log_satellite_run(r.fit, series=est.pd.series, projection=r.projection,
                                 search=r.search, run_name="satelite_pd_cartao")
    run = mlflow.get_run(run_id)
    print("run_id:", run_id[:12])
    print("params:", {k: run.data.params[k] for k in ("model", "kind", "link", "spec") if k in run.data.params})
    print("métricas:", {k: round(v, 4) for k, v in sorted(run.data.metrics.items())})

## 11. Fechamento — o mapa dos módulos

O subpacote cobre a arquitetura do guia (Tabela 2). A recomendação operacional: **subir a escada de
complexidade degrau a degrau, mantendo o degrau anterior como benchmark**, e uma **fonte única** de cenários e
projeção para todos os usos.

| Etapa (guia) | Onde no pacote |
|---|---|
| Contrato + dados sintéticos (§2.1, §7) | `RiskSeries`, `make_reference_study`, `simulate_pd/lgd/ccf_series` |
| Transformações (§2.2) | `transforms` (logit/probit, `vasicek_z`, defasagens, dummies) |
| Diagnóstico (§2.3, §4.2) | `diagnostics` (ADF/KPSS/PP, Ljung-Box, BG/BP/White, JB, ARCH-LM, VIF, Chow/CUSUM) |
| Modelo principal e benchmarks (§3.1–3.2) | `ARDL`, `ARIMA`, `RandomWalk`/`HistoricalMean`/`SeasonalNaive` |
| LGD/CCF (§3.5) | `BetaRegression`, `FractionalLogit`, `decompose_lgd` |
| Fator Z / longo prazo / painel (§3.3–3.7) | `VasicekZ`, `VARModel`/`VECMModel`, `PanelSatellite` |
| Seleção (§4) | `make_grid`, `search`, `walk_forward`, `diebold_mariano`, `compare` |
| Cenários (§5) | `Scenario`/`ScenarioSet`, `project`, `ecl_projection`, `standard_scenarios` |
| Pipeline + governança (§6) | `StudyConfig`/`run_study`, `report`, `tracking` |

**Próximos passos:** construir as séries reais por segmento a partir das bases contratuais (PySpark/Databricks),
listar as candidatas macro com mecanismo causal e sinal esperado, e conectar as projeções aos usos — ECL
(ponderação), estresse (trajetória adversa) e capital (ligação fator-macro).